# NB09 — Multicollinearity

> **When predictors are correlated, coefficients become unstable and uninterpretable.**


## What is multicollinearity?

When two or more predictors are strongly correlated with each other, the model can't separate their individual effects.

**Consequence:** large standard errors → wide CIs → statistically insignificant coefficients, even when the predictors genuinely explain y.

**Does NOT affect:** predictions, R², model fit.
**Does affect:** coefficient estimates, SEs, t-tests, interpretation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

np.random.seed(42)
n = 100

# Two correlated predictors
x1 = np.random.normal(0, 1, n)
x2 = 0.95 * x1 + np.random.normal(0, 0.3, n)   # highly correlated with x1
y  = 3*x1 + 2*x2 + np.random.normal(0, 1, n)

print(f"Correlation(x1, x2) = {np.corrcoef(x1, x2)[0,1]:.3f}")

# Fit model with both predictors
X_both = sm.add_constant(np.column_stack([x1, x2]))
res_both = sm.OLS(y, X_both).fit()
print("\n--- Model with BOTH x1 and x2 ---")
print(res_both.summary().tables[1])

# Fit models with one predictor each
X1 = sm.add_constant(x1)
X2 = sm.add_constant(x2)
res_x1 = sm.OLS(y, X1).fit()
res_x2 = sm.OLS(y, X2).fit()
print("\n--- Model with x1 only ---")
print(res_x1.summary().tables[1])
print("\n--- Model with x2 only ---")
print(res_x2.summary().tables[1])

print("\nNotice: individually x1 and x2 are significant,")
print("but together their SEs blow up due to multicollinearity.")


## VIF — Variance Inflation Factor

```
VIF_j = 1 / (1 − R²_j)
```

where R²_j is R² from regressing x_j on all other predictors.

| VIF | Interpretation |
|-----|---------------|
| 1 | No correlation with other predictors |
| 1–5 | Mild multicollinearity |
| 5–10 | Moderate — investigate |
| > 10 | Severe multicollinearity — action required |


In [ ]:
# Compute VIF from scratch
import numpy as np

def compute_vif(X_arr):
    """Compute VIF for each column in X_arr (no intercept column)."""
    n, k = X_arr.shape
    vifs  = []
    for j in range(k):
        y_j = X_arr[:, j]
        X_j = np.delete(X_arr, j, axis=1)
        X_j_d = np.column_stack([np.ones(n), X_j])
        beta = np.linalg.solve(X_j_d.T @ X_j_d, X_j_d.T @ y_j)
        yhat = X_j_d @ beta
        ss_res = np.sum((y_j - yhat)**2)
        ss_tot = np.sum((y_j - y_j.mean())**2)
        r2_j   = 1 - ss_res/ss_tot
        vif    = 1 / max(1 - r2_j, 1e-10)
        vifs.append(vif)
    return vifs

X_mat = np.column_stack([x1, x2])
vifs  = compute_vif(X_mat)
for name, v in zip(['x1', 'x2'], vifs):
    print(f"VIF({name}) = {v:.2f}")

print("\nVIF > 10 → severe multicollinearity confirmed")

# Cross-check with statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor
Xd = sm.add_constant(X_mat)
for i, name in enumerate(['const','x1','x2']):
    print(f"statsmodels VIF({name}) = {variance_inflation_factor(Xd, i):.2f}")


In [ ]:
# Correlation heatmap — another diagnostic
import seaborn as sns
from sklearn.datasets import fetch_california_housing
import pandas as pd

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['target'] = housing.target

corr = df.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation matrix — California Housing')
plt.tight_layout(); plt.show()
print("Off-diagonal values near ±1 → potential multicollinearity.")


## Fixes for multicollinearity

| Method | What it does |
|--------|-------------|
| Remove one of the correlated predictors | Simplest fix |
| PCA before regression | Create uncorrelated components |
| Ridge regression (L2) | Shrinks large coefficients, stable under collinearity |
| Lasso (L1) | Performs automatic variable selection |
| Collect more data | Reduces SEs without changing structure |

**Next:** NB10 — Polynomial regression and interaction terms.
